# French Enterprise Tutor — LoRA Fine-Tune of Gemma-2-9B (Unsloth)

Trains one LoRA adapter on the **1,542-row** `fr_v3_merged` dataset (French
path -- see `docs/architecture/rectified/analyze_05_french_finetune_plan.md`)
and exports it as a LoRA adapter (primary) plus a Q4_K_M GGUF for Ollama
(secondary). Read this cell before running anything.

**This notebook is adapted from `kaggle_finetune_v11.ipynb`** (the Darija /
Atlas-Chat-9B fine-tune). Sections 1, 4, 5, 6, 7 are reused unmodified where
the underlying mechanism is language-agnostic or Gemma-2-native (chat
template, masking, trainer, LoRA config) — see the per-section notes below
for exactly what changed and why.

## Run this with "Save & Run All (Commit)", NOT interactively

Same rule as the generation runs: an interactive session has a 60-minute idle
timeout that kills every process in the container. A committed run is
headless, has no idle timeout, and persists `/kaggle/working` as version
output.

**Settings:** Accelerator: **GPU T4 x2** (or P100) · Internet: **On** ·
Persistence: Variables and Files.

**Dataset upload must contain:** `fr_v3_merged/` (with `train.jsonl` +
`eval.jsonl`) and `app/` (used for the train/serve prompt parity assertion
against `SYSTEM_PROMPT_TEMPLATE_FR` — the run still works without it, with
that check skipped).

---

## What changed from the Darija notebook, and why

### Base model: `unsloth/gemma-2-9b`, not Atlas-Chat-9B

Stock Gemma-2-9B, not a Darija fine-tune of it — this is the French path's
own base per `analyze_05_french_finetune_plan.md` §"LoRA recipe": "Gemma-2-9B
is Atlas-Chat's base" — same architecture, same softcapping, same chat
template mechanics, different (absent) prior fine-tuning. `unsloth/gemma-2-9b`
is Unsloth's own un-gated mirror (confirmed 2026-08-04: no license-gate
requiring `HF_TOKEN`), matching the same "no token needed" constraint the
Atlas-Chat notebook relied on.

### Chat template (Sections 4-5): reused BYTE-IDENTICAL, not re-derived

`LESSONS_LEARNED.md` #4 / `MIGRATION_PLAN.md` P9: "reuse its chat-template +
byte-parity assertion cell UNMODIFIED — Gemma-2 has no system-role turn,
already solved there." Atlas-Chat-9B IS a Gemma-2-9B fine-tune, so its
system-merge-into-first-user-turn template, `<start_of_turn>`/`<end_of_turn>`
tokens, and the parity-proof mechanism apply to stock Gemma-2-9B without
modification. Do not touch Sections 4-5 without re-deriving the reason first.

### Dataset: `fr_v3_merged` (1,542 rows: 1,387 train / 155 eval), 8 components

`code_switching`, `darija_preservation`, `reasoning_preservation` are absent
by design (`FRENCH_COMPONENT_CONFIG` — no French analogue, or deliberately
dropped, see `analyze_05` §1 "Risk 1"). Train/serve parity check now compares
against `SYSTEM_PROMPT_TEMPLATE_FR`, not the Darija template.

### Script-gate direction is INVERTED for the generation smoke test

Darija's smoke test checks for Arabic-script *presence*. French's checks for
Arabic-script *absence outside citation spans* —
`has_arabic_outside_citations()` (`app/services/generate_training_data.py`),
the same function this session's audit used to verify F1's fix, reused here
against the fine-tuned model's own generations. This directly implements
`analyze_05` §4's acceptance gate: "`arabic_outside_citations` = 0 on 100% of
French turns."

### NEW — Section 10.5: base-vs-adapter comparison, built into this run

Not an afterthought. `docs/LESSONS_LEARNED.md` #1: "the exact discipline that
caught the original Darija citation-fabrication defect" — the trained
adapter must not fall below stock Gemma's own grounding floor on the same
prompts. Uses `model.disable_adapter()` (no second model load — same PEFT
wrapper, LoRA off vs. on) to generate from both on real grounded eval rows,
then checks for the specific worst-failure-mode pattern: the adapter citing
a reference number the source context does not contain, on a prompt where
the un-adapted base model did not fabricate one. See Section 10.5 below for
what it does and does not assert.

### Hyperparameters: UNCHANGED

`MIGRATION_PLAN.md` P9: "Do not change the LoRA hyperparameters without
cause." r=16/alpha=16/dropout=0, same 7 target modules, 2 epochs, batch 1 x
accum 16, lr 2e-4 cosine 3% warmup, `adamw_8bit`. See the original notebook's
own hyperparameter table for the reasoning — none of it was Darija-specific.

---

## Fail-fast order (unchanged from the Darija notebook)

| § | Step | Cost |
|---|---|---|
| 1 | env + single-GPU pin + install | ~5 min |
| 2 | dataset located, counted, structurally validated, parrot rows dropped | ~10 s |
| 3 | model download + 4-bit quantise | ~15-20 min (9B, smaller download than Atlas's bf16-only repo) |
| 4 | chat template installed + **parity assertion** | ~10 s |
| 5 | tokenise, length report, overlong rows dropped | ~1 min |
| 6 | LoRA attached, masking **verified and asserted** | ~1 min |
| 7 | trainer built + one-batch finite-loss smoke test | ~2 min |
| 8 | train | **~1-1.5 h** (1,387 rows vs. Darija's 2,757 — roughly half the steps) |
| 9 | per-component eval loss | ~3 min |
| 10 | generation smoke test | ~3 min |
| 10.5 | **base-vs-adapter comparison** | ~3 min |
| 11 | save, patch, package, GGUF | ~45 min |

**Expected wall clock ≈ 3-3.5 h.** `TRAIN_TIME_BUDGET_H` in §7 hard-stops
training so §11 is always reached.


## RESUME NOTEBOOK -- reads before running

This notebook resumes the French fine-tune from an already-completed adapter checkpoint
(`checkpoint-174`, 174/174 steps, 2/2 epochs) rather than retraining. The original run
(`darija-tutor-fr-finetune-v1`) trained successfully but crashed afterward on a missing
`app/` upload in that run's specific dataset version, before it ever reached the
post-training safety checks. See `MIGRATION_PLAN.md` for the full incident writeup.

**What changed vs. `kaggle_finetune_fr_v1.ipynb`:**
- Loads the base model + the already-trained LoRA adapter together from a new
  `darija-tutor-fr-checkpoint174` dataset input, instead of loading the base model fresh
  and training a new adapter from scratch.
- Skips the masking-probe sanity check, `Trainer` construction, the pre-train smoke
  forward-pass, and the actual `trainer.train()` call -- nothing here re-trains.
- Reconstructs training telemetry (eval-loss curve, steps completed) from
  `checkpoint-174/trainer_state.json` -- real numbers the original run saved before it
  crashed, honestly labeled as reconstructed rather than a fresh pre/post-training
  comparison.
- Adds the `HAVE_APP` guard that was missing on the cell that actually crashed, so this
  failure mode can't silently kill a future run the same way again.
- Everything from the per-component eval scoring onward -- sample generation, the smoke
  test, the base-vs-adapter citation-fabrication gate, merge, and GGUF export -- is
  unchanged and runs for real against the trained adapter for the first time.


## Section 1 — Environment

`CUDA_VISIBLE_DEVICES` is set **before** anything imports torch. Nothing in
this cell may import torch, directly or transitively — which is also why the
torch reinstall below can take effect at all.

In [ ]:
import time as _t

_PROGRESS_LOG = "/kaggle/working/PROGRESS.log"
_CRASH_LOG = "/kaggle/working/CRASH_TRACEBACK.txt"


def _mark(msg):
    with open(_PROGRESS_LOG, "a", encoding="utf-8") as _f:
        _f.write(_t.strftime("%H:%M:%S") + " " + msg + "\n")
    print("[progress]", msg)


_mark("notebook started")


_mark("entering cell 0")
try:
    import os

    # Unsloth OSS is single-GPU. Unpinned on 2xT4, HF Trainer wraps the model in
    # nn.DataParallel, which breaks Unsloth's patched kernels and gradient
    # checkpointing. Must happen before torch is imported by anything.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    os.environ["WANDB_DISABLED"] = "true"
    # Keep the ~18.5GB base-model download OFF /kaggle/working: that directory is
    # capped at 20GB and is saved as version output, so caching the model there
    # would blow the quota AND upload 18.5GB of unmodified base weights.
    os.environ["HF_HOME"] = "/root/hf_cache"

    import shutil
    import subprocess

    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version",
         "--format=csv"],
        capture_output=True, text=True).stdout)

    for path in ("/", "/kaggle/working"):
        total, used, free = shutil.disk_usage(path)
        print(f"disk {path:16s} free {free / 1e9:6.1f} GB / total {total / 1e9:6.1f} GB")

    # Base model is ~18.5GB of bf16 safetensors, quantised to 4-bit on the fly.
    assert shutil.disk_usage("/")[2] > 25e9, (
        "Less than 25GB free on the container disk; the base-model download will "
        "die partway through. Restart the session."
    )
    print("\nCUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
    _mark("completed cell 0")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 0 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 0: " + type(_e).__name__ + ": " + str(_e))
    raise


Versions are pinned to the combination Unsloth's own maintained Kaggle
Gemma-2-9B notebook uses. That is the path validated against Kaggle's current
image, so it is deliberately *not* "modernised" here. Output is captured
because pip emits several thousand lines; the tail and the resolved versions
are printed next.

A failed `!pip` does not raise, so the real safety net is the import
assertions in the following cell — not this one.

In [ ]:
_mark("about to enter pip-install cell (cell 1)")


In [ ]:
%%capture install_log
!pip install -q torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install -q unsloth
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
!pip install -q transformers==4.56.2
!pip install -q --no-deps trl==0.22.2

In [ ]:
_mark("survived pip-install cell (cell 1)")


In [ ]:
_mark("entering cell 2")
try:
    # Surface only the tail of the install log: enough to see a failure, not
    # enough to bury the rest of the run.
    log_text = getattr(install_log, "stdout", "") or str(install_log)
    tail = log_text.strip().splitlines()
    print("\n".join(tail[-15:]) if tail else "(pip was quiet)")

    # unsloth must be imported before transformers so its patches land.
    import unsloth
    from unsloth import FastLanguageModel
    import torch
    import transformers

    print("\ntorch       ", torch.__version__)
    print("transformers", transformers.__version__)
    print("unsloth     ", getattr(unsloth, "__version__", "unknown"))
    print("cuda        ", torch.version.cuda)

    assert torch.cuda.is_available(), "No CUDA device — the torch install broke the image."
    assert torch.cuda.device_count() == 1, (
        f"Expected exactly 1 visible GPU, got {torch.cuda.device_count()}. "
        "CUDA_VISIBLE_DEVICES was not applied before torch was imported."
    )

    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    SUPPORTS_BF16 = torch.cuda.is_bf16_supported()
    print(f"\nGPU: {GPU_NAME}  |  VRAM: {GPU_VRAM_GB:.1f} GB  |  bf16: {SUPPORTS_BF16}")
    print("precision ->", "bf16" if SUPPORTS_BF16 else "fp16 (T4/P100 have no bf16)")
    _mark("completed cell 2")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 2 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 2: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 2 — Locate and validate the dataset

Runs before the model download so a missing or malformed dataset costs 10
seconds instead of 25 minutes. The structural assertions encode exactly what
the masking in §6 relies on: system message first, strict alternation after
it, no Gemma control tokens smuggled inside content.

The third cell then drops **9 rows (0.34%)** whose assistant turn repeats a
chunk of the user's turn verbatim — `QUALITY_FLAGS.md` §7's "few-shot
parroting" defect, found while verifying the masking against this exact
dataset. Nine rows is too few to matter statistically and too damaging to
keep: echo-then-answer is the most visible way a tutor reads as broken, and
LoRA at r=16 will happily learn a distinctive surface pattern from 9
examples.

In [ ]:
_mark("entering cell 3")
try:
    from pathlib import Path

    def discover_src(marker="fr_v3_merged/train.jsonl"):
        """Find the uploaded dataset root under /kaggle/input by locating a known
        file inside it, instead of assuming a fixed mount path."""
        for root, dirs, files in os.walk("/kaggle/input"):
            if (Path(root) / marker).is_file():
                return Path(root)
        return None

    SRC = discover_src()
    if SRC is None:
        print("Could not auto-locate the dataset under /kaggle/input. Contents:")
        for root, dirs, files in os.walk("/kaggle/input"):
            depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
            print("  " * depth + root)
            for f in files[:5]:
                print("  " * (depth + 1) + f)
            if depth > 4:
                break
        raise SystemExit("Set SRC manually from the listing above, then re-run this cell.")

    DATA_DIR = SRC / "fr_v3_merged"
    TRAIN_JSONL = DATA_DIR / "train.jsonl"
    EVAL_JSONL = DATA_DIR / "eval.jsonl"
    for p in (TRAIN_JSONL, EVAL_JSONL):
        assert p.is_file(), f"{p} missing from the uploaded Kaggle Dataset."

    print("SRC      =", SRC)
    print("DATA_DIR =", DATA_DIR)

    # ---------------------------------------------------------------------------
    # ROOT-CAUSE FIX (kernel v5). The original line here was:
    #     APP_DIR = SRC / "app"
    # but SRC is the directory CONTAINING fr_v3_merged -- i.e. <dataset_root>/data --
    # while app/ is a SIBLING of data/, at <dataset_root>/app. So SRC/"app" resolved to
    # <dataset_root>/data/app, a path that cannot exist, and HAVE_APP was False on
    # EVERY run (including the original 3h one) no matter how correct the upload was.
    # Every "app/ not uploaded" failure traces back to this line, not to the dataset.
    # Locate the package directly by its own contents instead of by assumed position.
    # ---------------------------------------------------------------------------
    import sys

    def discover_app():
        """Find the app/ package anywhere under /kaggle/input by its contents."""
        for root, dirs, files in os.walk("/kaggle/input"):
            p = Path(root)
            if p.name == "app" and (p / "services" / "citations.py").is_file():
                return p
        return None

    APP_DIR = discover_app()
    HAVE_APP = APP_DIR is not None
    if HAVE_APP:
        shutil.copytree(str(APP_DIR), "/kaggle/working/app", dirs_exist_ok=True)
        print("copied app/ for the parity check, from", APP_DIR)
    else:
        print("app/ genuinely not found anywhere under /kaggle/input. Tree:")
        for root, dirs, files in os.walk("/kaggle/input"):
            depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
            if depth <= 2:
                print("  " * depth + root)

    os.chdir("/kaggle/working")
    # make `import app...` resolve regardless of how the kernel sets up sys.path
    if "/kaggle/working" not in sys.path:
        sys.path.insert(0, "/kaggle/working")

    # Fail loudly and immediately rather than 8 minutes later at the safety gate.
    assert HAVE_APP, (
        "app/ not found under /kaggle/input -- the base-vs-adapter citation-fabrication "
        "gate cannot run without it. Fix the dataset before spending GPU time."
    )
    import app.services.citations as _probe
    print("import app.services.citations OK ->", _probe.__file__)

    # app.services.llm -> app.config -> pydantic_settings. The parity check below
    # try/excepts this import, so a missing dep would silently SKIP the check rather
    # than fail it. Install it if absent so the check actually runs. (Verified
    # locally: PRODUCTION_SYSTEM_PROMPT_TEMPLATE_FR == SYSTEM_PROMPT_TEMPLATE_FR and
    # 0/1542 dataset rows mismatch, so this check is expected to PASS, not blow up.)
    try:
        import pydantic_settings  # noqa: F401
    except ImportError:
        import subprocess
        print("installing pydantic-settings for the parity check ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "pydantic-settings"],
            check=False,
        )

    _mark("completed cell 3")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 3 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 3: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 4")
try:
    import json
    from collections import Counter

    def load_jsonl(path):
        rows = []
        with open(path, "r", encoding="utf-8") as fh:
            for lineno, line in enumerate(fh, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise SystemExit(f"{path}:{lineno} is not valid JSON: {e}")
        return rows

    train_rows = load_jsonl(TRAIN_JSONL)
    eval_rows = load_jsonl(EVAL_JSONL)
    print(f"train.jsonl {len(train_rows):5d} rows")
    print(f"eval.jsonl  {len(eval_rows):5d} rows")
    print(f"total       {len(train_rows) + len(eval_rows):5d} rows")

    CONTROL_TOKENS = ("<start_of_turn>", "<end_of_turn>", "<bos>", "<eos>", "<pad>")

    def validate(rows, label):
        """Check every invariant the masking in Section 6 depends on."""
        problems = Counter()
        for r in rows:
            ms = r.get("messages") or []
            if not ms or ms[0]["role"] != "system":
                problems["no leading system message"] += 1
                continue
            if any(m["role"] == "system" for m in ms[1:]):
                problems["system message after position 0"] += 1
            body = ms[1:]
            expected = ["user" if i % 2 == 0 else "assistant" for i in range(len(body))]
            if [m["role"] for m in body] != expected:
                problems["roles do not alternate user/assistant"] += 1
            if not body or body[-1]["role"] != "assistant":
                problems["does not end on an assistant turn"] += 1
            for m in ms:
                if not m.get("content", "").strip():
                    problems["empty message content"] += 1
                if any(t in m["content"] for t in CONTROL_TOKENS):
                    problems["Gemma control token inside content"] += 1
        print(f"\n{label}: {len(rows)} rows, {sum(problems.values())} problems")
        for k, v in problems.items():
            print(f"  {k}: {v}")
        return problems

    p1 = validate(train_rows, "train")
    p2 = validate(eval_rows, "eval")
    assert not (p1 + p2), (
        "Dataset violates a structural invariant the masking depends on (see "
        "above). Do not train on this; fix the export first."
    )

    print("\nby component:")
    all_rows = train_rows + eval_rows
    for comp, n in Counter(r["component"] for r in all_rows).most_common():
        tr = sum(1 for r in train_rows if r["component"] == comp)
        ev = sum(1 for r in eval_rows if r["component"] == comp)
        print(f"  {comp:24s} {n:5d}  (train {tr:5d} / eval {ev:4d})")

    # Eval must cover every component, or the per-component loss table in Section 9
    # is blind exactly where it matters most (the thin components).
    missing = set(r["component"] for r in train_rows) - set(r["component"] for r in eval_rows)
    assert not missing, f"components absent from eval.jsonl: {missing}"

    n_sys = len({r["messages"][0]["content"] for r in all_rows})
    print(f"\ndistinct system prompts: {n_sys} across {len(all_rows)} rows "
          f"(~{len(all_rows) / n_sys:.0f}x repetition -> masking is mandatory)")
    _mark("completed cell 4")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 4 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 4: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 5")
try:
    # Parrot filter -- standing safety net carried over from the Darija
    # notebook (QUALITY_FLAGS.md section 7's "few-shot parroting" failure
    # mode: assistant echoes a chunk of the user's turn verbatim before
    # answering). Not measured against fr_v3_merged before this run -- this
    # cell IS that measurement, not a re-statement of a known rate. The
    # threshold is deliberately conservative: 50 consecutive characters is
    # far beyond coincidental overlap in French too.
    PARROT_CHARS = 50

    def is_parrot(row):
        users = [m["content"].strip() for m in row["messages"] if m["role"] == "user"]
        answers = " ".join(m["content"] for m in row["messages"]
                           if m["role"] == "assistant")
        return any(u[:PARROT_CHARS] in answers for u in users if len(u) >= PARROT_CHARS)

    _before = (len(train_rows), len(eval_rows))
    parrot_rows = [r for r in train_rows + eval_rows if is_parrot(r)]
    train_rows = [r for r in train_rows if not is_parrot(r)]
    eval_rows = [r for r in eval_rows if not is_parrot(r)]
    all_rows = train_rows + eval_rows

    print(f"parroting rows dropped: {len(parrot_rows)} "
          f"({len(parrot_rows) / sum(_before):.2%})")
    for comp, k in Counter(r["component"] for r in parrot_rows).most_common():
        print(f"  {comp:24s} {k}")
    print(f"train {_before[0]} -> {len(train_rows)}   eval {_before[1]} -> {len(eval_rows)}")
    assert len(parrot_rows) / sum(_before) < 0.05, (
        "More than 5% of rows are parroting the user. That is a generation-prompt "
        "defect, not something to filter away — fix it upstream and regenerate."
    )
    _mark("completed cell 5")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 5 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 5: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 6")
try:
    # Train/serve prompt parity. The system prompts baked into this dataset must be
    # renderings of the same template llm.py sends in production for French
    # queries; the project has held PRODUCTION_SYSTEM_PROMPT_TEMPLATE_FR ==
    # SYSTEM_PROMPT_TEMPLATE_FR as an invariant since this session added the
    # regression test (tests/test_generation_gates.py). This is where a silent
    # drift would surface before six hours of training on a stale prompt.
    import sys
    sys.path.insert(0, "/kaggle/working")
    PARITY_CHECKED = False
    if HAVE_APP:
        try:
            from app.services.generate_training_data import PRODUCTION_SYSTEM_PROMPT_TEMPLATE_FR
            from app.services.llm import SYSTEM_PROMPT_TEMPLATE_FR
        except Exception as e:
            print(f"parity check SKIPPED ({type(e).__name__}: {e})")
        else:
            assert PRODUCTION_SYSTEM_PROMPT_TEMPLATE_FR == SYSTEM_PROMPT_TEMPLATE_FR, (
                "generate_training_data.PRODUCTION_SYSTEM_PROMPT_TEMPLATE_FR has "
                "drifted from llm.SYSTEM_PROMPT_TEMPLATE_FR — the dataset was built "
                "with one template and production serves the other."
            )
            head = PRODUCTION_SYSTEM_PROMPT_TEMPLATE_FR.split("{domain}")[0]
            bad = [r for r in all_rows if not r["messages"][0]["content"].startswith(head)]
            assert not bad, f"{len(bad)} rows carry a system prompt from a different template"
            PARITY_CHECKED = True
            print("prompt parity OK: dataset system prompts match the production")
            print("French template, and generate_training_data == llm.py")
    else:
        print("parity check SKIPPED (app/ not uploaded)")

    _mark("completed cell 6")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 6 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 6: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 3 — Load Gemma-2-9B in 4-bit

`unsloth/gemma-2-9b` is Unsloth's own un-gated mirror of stock Gemma-2-9B —
confirmed 2026-08-04 not to require a license-gate `HF_TOKEN` (unlike
`google/gemma-2-9b` on the canonical repo), matching the same "no token
needed" property the Atlas-Chat-9B base relied on. Unsloth publishes a
pre-quantized 4-bit bnb variant for this model, so the ~18.5GB bf16 download
this section historically needed for Atlas-Chat may be smaller here.

**This cell takes ~15-20 minutes.**

`dtype=None` lets Unsloth pick fp16 on T4/P100 and bf16 on Ampere+. Gemma-2's
`attn_logit_softcapping=50` / `final_logit_softcapping=30` are why this needs
Unsloth on a T4 at all: softcapping is incompatible with FlashAttention-2, and
naive fp16 softcapping overflows. Identical constraint to Atlas-Chat, since
Atlas-Chat inherits this from being a Gemma-2-9B fine-tune itself.


In [ ]:
_mark("entering cell 7")
try:
    import time

    BASE_MODEL = "unsloth/gemma-2-9b"  # kept for report metadata; not loaded fresh
    MAX_SEQ_LENGTH = 4096   # see Section 5 for the measurement behind this


    def discover_adapter_src(marker="adapter_config.json"):
        """Find the uploaded checkpoint-174 adapter dataset under /kaggle/input,
        the same way discover_src() locates the training data above."""
        for root, dirs, files in os.walk("/kaggle/input"):
            if (Path(root) / marker).is_file():
                return Path(root)
        return None


    ADAPTER_SRC = discover_adapter_src()
    if ADAPTER_SRC is None:
        print("Could not auto-locate the checkpoint-174 adapter under /kaggle/input. Contents:")
        for root, dirs, files in os.walk("/kaggle/input"):
            depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
            print("  " * depth + root)
            for f in files[:5]:
                print("  " * (depth + 1) + f)
            if depth > 4:
                break
        raise SystemExit("Set ADAPTER_SRC manually from the listing above, then re-run this cell.")
    print("ADAPTER_SRC =", ADAPTER_SRC)

    # RESUME (see MIGRATION_PLAN.md): the original run trained this adapter to
    # completion -- 174/174 steps, 2/2 epochs -- then crashed on a missing app/
    # upload before it ever reached the post-training safety checks below.
    # Nothing in this notebook re-trains; it loads the base model AND the
    # already-trained LoRA adapter together (Unsloth auto-resolves
    # base_model_name_or_path from adapter_config.json) and re-runs only what
    # never got to execute: per-component eval, the base-vs-adapter fabrication
    # gate, merge, and GGUF export.
    t0 = time.time()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(ADAPTER_SRC),
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,          # auto: fp16 on T4/P100, bf16 on Ampere+
        load_in_4bit=True,
    )
    print(f"\nloaded base + trained adapter in {(time.time() - t0) / 60:.1f} min")
    print("arch         ", model.config.architectures)
    print("layers       ", model.config.num_hidden_layers)
    print("hidden       ", model.config.hidden_size)
    print("vocab        ", model.config.vocab_size)
    print("compute dtype", model.dtype)
    print(f"VRAM after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, "
          f"{torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")

    # Real training telemetry the original run saved before it crashed --
    # reconstructed from checkpoint-174/trainer_state.json, not fabricated.
    CKPT_TRAINER_STATE = json.loads((ADAPTER_SRC / "trainer_state.json").read_text())
    print(f"\ncheckpoint trainer_state: step {CKPT_TRAINER_STATE['global_step']}"
          f"/{CKPT_TRAINER_STATE['max_steps']}, epoch {CKPT_TRAINER_STATE['epoch']:.3f}")

    _mark("completed cell 7")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 7 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 7: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 4 — Install a system-aware chat template, and prove it matches

Atlas-Chat's stock template raises on `system`. This installs a replacement
that merges the system message into the first user turn, keeping everything
else byte-identical to Gemma-2's format (`| trim` on content, `<bos>` prefix,
`<end_of_turn>\n` terminators, `assistant` renamed to `model`).

The assertion at the bottom is the load-bearing line of the whole notebook: it
proves the encoder used for **training** and the template shipped to
**production** emit the same bytes. Edit one without the other and the run
stops here instead of six hours later.

In [ ]:
_mark("entering cell 8")
try:
    # Merges `system` into the first user turn. Otherwise identical to Atlas-Chat's
    # stock Gemma-2 template, alternation guard included. Raw strings keep the
    # backslash-n as Jinja source (Jinja unescapes it), matching upstream's form.
    # Content and terminator are emitted as separate {{ }} statements rather than
    # `content | trim + '<end_of_turn>'` so filter-vs-plus precedence cannot bite.
    SYSTEM_JOIN = "\n\n"

    CHAT_TEMPLATE = (
        "{{ bos_token }}"
        "{%- if messages[0]['role'] == 'system' -%}"
        "{%- set system_message = messages[0]['content'] | trim -%}"
        "{%- set loop_messages = messages[1:] -%}"
        "{%- else -%}"
        "{%- set system_message = '' -%}"
        "{%- set loop_messages = messages -%}"
        "{%- endif -%}"
        "{%- for message in loop_messages -%}"
        "{%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}"
        "{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}"
        "{%- endif -%}"
        "{%- if message['role'] == 'assistant' -%}"
        "{%- set role = 'model' -%}"
        "{%- else -%}"
        "{%- set role = message['role'] -%}"
        "{%- endif -%}"
        r"{{ '<start_of_turn>' + role + '\n' }}"
        "{%- if loop.first and system_message -%}"
        r"{{ system_message + '\n\n' }}"
        "{%- endif -%}"
        "{{ message['content'] | trim }}"
        r"{{ '<end_of_turn>\n' }}"
        "{%- endfor -%}"
        "{%- if add_generation_prompt -%}"
        r"{{ '<start_of_turn>model\n' }}"
        "{%- endif -%}"
    )

    tokenizer.chat_template = CHAT_TEMPLATE

    EOT_ID = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    BOS_ID = tokenizer.bos_token_id
    assert EOT_ID is not None and EOT_ID != tokenizer.unk_token_id, \
        "<end_of_turn> is not a real token in this tokenizer"
    print("bos          ", tokenizer.bos_token, BOS_ID)
    print("eos          ", tokenizer.eos_token, tokenizer.eos_token_id)
    print("pad          ", tokenizer.pad_token, tokenizer.pad_token_id)
    print("<end_of_turn>", EOT_ID)
    _mark("completed cell 8")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 8 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 8: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 9")
try:
    def split_system(messages):
        """Return (system_text_or_None, remaining_messages)."""
        ms = list(messages)
        if ms and ms[0]["role"] == "system":
            return ms[0]["content"].strip(), ms[1:]
        return None, ms


    def render(messages):
        """Render to training text. Deliberately WITHOUT <bos>: the tokenizer adds
        exactly one, and a literal <bos> here would produce the classic double-BOS
        that quietly degrades Gemma fine-tunes."""
        sys_txt, body = split_system(messages)
        out = []
        for i, m in enumerate(body):
            role = "model" if m["role"] == "assistant" else "user"
            content = m["content"].strip()
            if i == 0 and sys_txt:
                content = sys_txt + SYSTEM_JOIN + content
            out.append(f"<start_of_turn>{role}\n{content}<end_of_turn>\n")
        return "".join(out)


    # PARITY PROOF: our renderer vs the template we ship to production.
    probe_rows = train_rows[:400] + eval_rows[:200]
    mismatch = 0
    for r in probe_rows:
        ours = tokenizer.bos_token + render(r["messages"])
        theirs = tokenizer.apply_chat_template(r["messages"], tokenize=False)
        if ours != theirs:
            mismatch += 1
            if mismatch == 1:
                print("FIRST MISMATCH\n--- render() ---")
                print(repr(ours[:400]))
                print("--- chat_template ---")
                print(repr(theirs[:400]))
    assert mismatch == 0, (
        f"{mismatch}/{len(probe_rows)} rows render differently through render() "
        "than through the chat_template being shipped. Training format and serving "
        "format have diverged — fix before training."
    )
    print(f"parity OK: render() == chat_template on {len(probe_rows)} rows")

    print("\n--- rendered example (newlines shown as \\n) ---")
    ex = render(train_rows[0]["messages"])
    print(ex[:500].replace("\n", "\\n\n"))
    print("   ...")
    print(ex[-240:].replace("\n", "\\n\n"))
    _mark("completed cell 9")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 9 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 9: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 5 — Tokenise, measure real lengths, drop (never truncate) overlong rows

`MAX_SEQ_LENGTH` was chosen from character counts before a tokenizer was
available; this is where it gets checked against real token counts. Rows
longer than the limit are **dropped rather than truncated** — truncation cuts
an assistant answer mid-sentence and removes its `<end_of_turn>`, teaching the
model both to ramble and not to stop. Losing a handful of rows is far cheaper
than that.

In [ ]:
_mark("entering cell 10")
try:
    def encode(messages):
        """Tokenise one row, masking everything except the assistant bodies.

        Each assistant's <end_of_turn> is deliberately INSIDE the unmasked span:
        that token is how the model learns to stop, and Atlas-Chat's
        generation_config does not list it as EOS (patched in Section 11)."""
        sys_txt, body = split_system(messages)
        ids = [BOS_ID]
        labels = [-100]
        for i, m in enumerate(body):
            role = "model" if m["role"] == "assistant" else "user"
            content = m["content"].strip()
            if i == 0 and sys_txt:
                content = sys_txt + SYSTEM_JOIN + content
            head = tokenizer(f"<start_of_turn>{role}\n",
                             add_special_tokens=False)["input_ids"]
            tail = tokenizer(f"{content}<end_of_turn>\n",
                             add_special_tokens=False)["input_ids"]
            ids += head + tail
            # The turn header is always context; the body is a target only on
            # model turns.
            labels += [-100] * len(head)
            labels += tail if role == "model" else [-100] * len(tail)
        assert len(ids) == len(labels)
        return ids, labels


    # Piecewise tokenisation must still decode to the same text the shipped
    # template produces — this catches any seam artefact between head and body.
    for r in (train_rows[:50] + eval_rows[:20]):
        ids, _ = encode(r["messages"])
        assert tokenizer.decode(ids) == tokenizer.apply_chat_template(
            r["messages"], tokenize=False
        ), "encode() does not round-trip to the shipped chat_template"
    print("encode() round-trips to the shipped chat_template")
    _mark("completed cell 10")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 10 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 10: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 11")
try:
    import numpy as np

    def build(rows, label):
        kept, dropped = [], []
        for i, r in enumerate(rows):
            ids, labels = encode(r["messages"])
            rec = {
                "input_ids": ids,
                "labels": labels,
                "attention_mask": [1] * len(ids),
                "component": r["component"],
                "src_idx": i,
                "n_tokens": len(ids),
                "n_target": sum(1 for x in labels if x != -100),
            }
            (dropped if len(ids) > MAX_SEQ_LENGTH else kept).append(rec)

        lens = np.array([r["n_tokens"] for r in kept + dropped])
        print(f"\n{label}: {len(rows)} rows")
        print(f"  tokens  min {lens.min():5d}  p50 {int(np.percentile(lens, 50)):5d}"
              f"  p90 {int(np.percentile(lens, 90)):5d}"
              f"  p99 {int(np.percentile(lens, 99)):5d}  max {lens.max():5d}")
        tgt = np.array([r["n_target"] / r["n_tokens"] for r in kept])
        print(f"  trainable token share: p50 {np.percentile(tgt, 50):.0%}"
              f"  min {tgt.min():.0%}  max {tgt.max():.0%}")
        print(f"  dropped for exceeding MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}: {len(dropped)}")
        for d in dropped:
            print(f"    - {d['component']} at {d['n_tokens']} tokens")
        return kept


    train_enc = build(train_rows, "train")
    eval_enc = build(eval_rows, "eval")

    # Losing a lot here means MAX_SEQ_LENGTH is wrong, not that the data is bad —
    # stop rather than silently training on a length-biased subset.
    n_in = len(train_rows) + len(eval_rows)
    drop_rate = 1 - (len(train_enc) + len(eval_enc)) / n_in
    assert drop_rate < 0.02, (
        f"{drop_rate:.1%} of rows exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. Raise it "
        "(Gemma-2 supports 8192) rather than dropping this much data."
    )
    print(f"\nkept: train {len(train_enc)} / eval {len(eval_enc)}  "
          f"(dropped {drop_rate:.2%} overall)")

    total_target_tokens = sum(r["n_target"] for r in train_enc)
    print(f"trainable tokens per epoch: {total_target_tokens:,}")
    _mark("completed cell 11")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 11 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 11: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 12")
try:
    from datasets import Dataset

    COLS = ["input_ids", "labels", "attention_mask"]
    train_ds = Dataset.from_list([{k: r[k] for k in COLS} for r in train_enc])
    eval_ds = Dataset.from_list([{k: r[k] for k in COLS} for r in eval_enc])
    print(train_ds)
    print(eval_ds)
    _mark("completed cell 12")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 12 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 12: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 6 — Attach LoRA, then verify the masking is actually right

The printout below is worth reading in the committed log: for one real row it
shows exactly which tokens carry gradient. The system prompt and the user's
question must be **absent**; the assistant's answer and its `<end_of_turn>`
must be **present**. The assertions enforce both, so a masking regression
fails here in seconds rather than surfacing as a model that recites its own
system prompt three hours later.

In [ ]:
_mark("entering cell 13")
try:
    # RESUME: no fresh LoRA init -- model already carries the trained
    # checkpoint-174 adapter loaded above. adapter_config.json has
    # "inference_mode": true, so requires_grad is False on the LoRA params here
    # by design (we are not training); that is expected and does not affect
    # model.disable_adapter() or generation below.
    #
    # `import math` moved here: the original notebook's only import of it lived
    # inside the Trainer-setup cell (deleted below, since no Trainer is needed
    # without a live training loop) but math.exp() is also used much later in
    # the per-component eval-loss cell -- an orphaned dependency the first
    # resume attempt crashed on (NameError: name 'math' is not defined).
    import math
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} = {100 * trainable / total:.3f}% "
          f"(0 is expected in inference mode)")

    _mark("completed cell 13")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 13 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 13: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 7 — Trainer

Argument names are resolved by signature introspection
(`eval_strategy`/`evaluation_strategy`, `processing_class`/`tokenizer`),
because those renames are the most common reason a pinned notebook stops
running six months later.

`TRAIN_TIME_BUDGET_H` installs a callback that stops training cleanly if the
budget is exceeded, so §11's save and export always run.

## Section 8 — Train

Watch two things in the log: `loss` should fall then flatten, and the periodic
`eval_loss` should fall then flatten *with it*. Eval loss turning back up
while train loss keeps falling is overfitting — in that case the epoch-1
checkpoint under `/kaggle/working/checkpoints` is the one you want, not the
final adapter. §8's tail prints which step had the best eval loss.

In [ ]:
_mark("entering cell 14")
try:
    # RESUME: no training happens in this notebook. This reconstructs the
    # telemetry the skipped training cells would have produced, entirely from
    # checkpoint-174/trainer_state.json -- real numbers the original run saved
    # before crashing, not fabricated. Note: the original notebook's cell 17 ran
    # an explicit trainer.evaluate() BEFORE training started to get a true
    # pre-training baseline; that ad-hoc call was never written into
    # trainer_state.json (only the Trainer's own periodic logging is), so that
    # exact baseline number is genuinely lost. What follows uses the earliest
    # and latest eval checkpoints actually on record instead, labeled honestly
    # as such rather than mislabeled as a pre-training baseline.
    EPOCHS = 2
    BATCH = 1
    ACCUM = 16
    LR = 2e-4

    hist = [h for h in CKPT_TRAINER_STATE.get("log_history", []) if "eval_loss" in h]
    train_loss_entries = [
        h for h in CKPT_TRAINER_STATE.get("log_history", [])
        if "loss" in h and "eval_loss" not in h
    ]

    steps_completed = CKPT_TRAINER_STATE["global_step"]
    steps_planned = CKPT_TRAINER_STATE["max_steps"]
    epoch_reached = CKPT_TRAINER_STATE["epoch"]

    first_recorded_eval_loss = hist[0]["eval_loss"] if hist else None
    final_recorded_eval_loss = hist[-1]["eval_loss"] if hist else None
    final_train_loss = train_loss_entries[-1]["loss"] if train_loss_entries else None
    best = min(hist, key=lambda h: h["eval_loss"]) if hist else None
    train_minutes = None  # wall-clock time was in the lost log, not recoverable

    print(f"steps {steps_completed}/{steps_planned}   epoch {epoch_reached:.3f}/{EPOCHS}")
    print(f"eval_loss: first-recorded (step {hist[0]['step'] if hist else '?'}) "
          f"{first_recorded_eval_loss} -> last-recorded (step "
          f"{hist[-1]['step'] if hist else '?'}) {final_recorded_eval_loss}")
    print(f"final logged train loss (step {train_loss_entries[-1]['step'] if train_loss_entries else '?'}): "
          f"{final_train_loss}")
    print("\neval curve (from checkpoint-174/trainer_state.json):")
    for h in hist:
        print(f"  step {h.get('step', 0):4d}  eval_loss {h['eval_loss']:.4f}")
    if best is not None and best is not hist[-1]:
        print(f"\n  NOTE: best recorded eval_loss was {best['eval_loss']:.4f} at step "
              f"{best.get('step')}, not at the end ({hist[-1]['eval_loss']:.4f}).")
    else:
        print("\n  eval_loss was still improving at the last recorded step -- "
              "consistent with a healthy, non-overfit run.")

    assert steps_completed == steps_planned, (
        "checkpoint-174 does not show a completed run -- do not treat this as "
        "the final adapter."
    )
    print("\nconfirmed: checkpoint-174 completed all planned steps and epochs.")

    _mark("completed cell 14")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 14 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 14: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 9 — Per-component eval loss

Aggregate eval loss hides the thing that matters most here: this is **one**
adapter carrying eight different behaviours (`FRENCH_COMPONENT_CONFIG`).
Two of them — `structured_explanation` and `quiz_generation` — are known to
sit near a corpus-size dedup ceiling (`data/fr_v3_merged` has them at ~150
rows each against a 300-row design target, a decision accepted this session
rather than chased further) -- watch this table for whether that shortfall
shows up as measurably worse per-token loss, not just a smaller row count.

A component whose loss sits far above the others is underfit. This table is
the evidence for whether any component needs a targeted top-up before this
adapter ships.

Loss is taken from the model's own fused cross-entropy (`labels=` on the
forward pass) rather than by materialising logits: at 4096 positions × 256k
vocab an fp32 logits tensor is ~4 GB and would OOM a T4.


In [ ]:
_mark("entering cell 15")
try:
    from collections import defaultdict

    model.eval()
    per_comp = defaultdict(lambda: [0.0, 0])   # component -> [loss_sum, n_target_tokens]

    with torch.no_grad():
        for k, rec in enumerate(eval_enc):
            ids = torch.tensor([rec["input_ids"]], device="cuda")
            lab = torch.tensor([rec["labels"]], device="cuda")
            out = model(input_ids=ids, attention_mask=torch.ones_like(ids), labels=lab)
            # HF returns mean CE over non-ignored SHIFTED labels, so multiply back
            # up by that count to get a sum we can pool across rows.
            n = int((lab[:, 1:] != -100).sum())
            per_comp[rec["component"]][0] += float(out.loss) * n
            per_comp[rec["component"]][1] += n
            if (k + 1) % 50 == 0:
                print(f"  scored {k + 1}/{len(eval_enc)}")
            del out

    rows_out = []
    for comp, (s, n) in per_comp.items():
        n_eval = sum(1 for r in eval_enc if r["component"] == comp)
        n_train = sum(1 for r in train_enc if r["component"] == comp)
        rows_out.append((comp, n_train, n_eval, s / max(n, 1)))
    rows_out.sort(key=lambda t: t[3], reverse=True)

    print()
    print(f"{'component':24s} {'train':>6s} {'eval':>5s} {'loss/token':>11s} {'ppl':>8s}")
    print("-" * 60)
    for comp, n_train, n_eval, loss in rows_out:
        print(f"{comp:24s} {n_train:6d} {n_eval:5d} {loss:11.4f} "
              f"{math.exp(min(loss, 20)):8.2f}")
    print("-" * 60)
    overall = (sum(s for s, _ in per_comp.values())
               / sum(n for _, n in per_comp.values()))
    print(f"{'OVERALL':24s} {len(train_enc):6d} {len(eval_enc):5d} {overall:11.4f} "
          f"{math.exp(min(overall, 20)):8.2f}")

    flagged = [r for r in rows_out if r[3] > 1.6 * overall]
    if flagged:
        for comp, n_train, n_eval, loss in flagged:
            print(f"\nUNDERFIT: '{comp}' at {loss:.3f} loss/token is "
                  f"{loss / overall:.2f}x the overall mean on only {n_train} train "
                  f"rows. Top that component up before shipping this adapter.")
    else:
        print(f"\nNo component exceeds 1.6x the mean loss — the adapter learned all "
              f"{len(rows_out)} behaviours at comparable quality.")
    _mark("completed cell 15")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 15 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 15: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 10 — Generation smoke test

Loss cannot tell you whether the model still writes Arabic-script Darija,
still code-switches French, still emits parseable quiz JSON, or stops cleanly.
This generates from real eval prompts and checks all four.

In [ ]:
_mark("entering cell 16")
try:
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"    # correct side for generation

    STOP_IDS = sorted({tokenizer.eos_token_id, EOT_ID})
    print("stop ids:", STOP_IDS)


    def ask(row, max_new_tokens=320):
        """Prompt the model with a real eval row's system + first user turn."""
        msgs = row["messages"][:2]
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer([text], return_tensors="pt",
                           add_special_tokens=False).to("cuda")
        gen = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id,
        )
        return tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()


    ARABIC = range(0x0600, 0x0700)
    samples = {}
    for comp in ("socratic", "grounded_refusal", "quiz_generation",
                 "structured_explanation", "learner_adaptation"):
        row = next((r for r in eval_rows if r["component"] == comp), None)
        if row is None:
            continue
        ans = ask(row)
        samples[comp] = ans
        n_ar = sum(1 for ch in ans if ord(ch) in ARABIC)
        print("=" * 70)
        print(f"[{comp}]  {len(ans)} chars, {n_ar} Arabic-script chars")
        print("USER :", row["messages"][1]["content"][:160])
        print("MODEL:", ans[:700])
        print()
    _mark("completed cell 16")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 16 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 16: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 17")
try:
    # Four things the loss curve cannot see. French mode's script gate is the
    # INVERSE of Darija's: correct output is Latin-script, with Arabic permitted
    # only inside a verbatim legal citation (analyze_05 §4: "arabic_outside_
    # citations = 0 on 100% of French turns"). has_arabic_outside_citations() is
    # the exact function this session's F1 audit used to verify the citation-
    # injection fix -- reused here against the fine-tuned model's own output,
    # not just against training rows.
    issues = []

    if HAVE_APP:
        from app.services.generate_training_data import has_arabic_outside_citations
    else:
        has_arabic_outside_citations = None
        issues.append("app/ not uploaded -- cannot run the arabic-outside-citations "
                       "script gate, this smoke test is incomplete")

    for comp, ans in samples.items():
        if comp == "quiz_generation":
            continue
        if not ans:
            issues.append(f"{comp}: empty generation")
            continue
        if has_arabic_outside_citations is not None and has_arabic_outside_citations(ans):
            issues.append(f"{comp}: Arabic-script characters outside a citation span "
                           f"-- French-mode script gate violated")

    if "quiz_generation" in samples:
        raw = samples["quiz_generation"].strip()
        for fence in ("```json", "```"):
            raw = raw.removeprefix(fence)
        raw = raw.removesuffix("```").strip()
        try:
            parsed = json.loads(raw)
            n_q = len(parsed.get("questions", []))
            print(f"quiz JSON parses: {n_q} questions")
            if n_q == 0:
                issues.append("quiz_generation: parsed but zero questions")
        except json.JSONDecodeError as e:
            issues.append(f"quiz_generation: JSON does not parse ({e})")

    # Did it stop, or start role-playing the user's next turn?
    for comp, ans in samples.items():
        if "<start_of_turn>" in ans or "CONTEXTE :" in ans:
            issues.append(f"{comp}: leaked a turn marker / context — stop-token problem")

    CJK = list(range(0x3000, 0xA000)) + list(range(0xAC00, 0xD800))
    CJK_SET = set(CJK)
    for comp, ans in samples.items():
        if any(ord(ch) in CJK_SET for ch in ans):
            issues.append(f"{comp}: CJK contamination")

    print("\n" + ("SMOKE TEST CLEAN" if not issues else "SMOKE TEST ISSUES:"))
    for i in issues:
        print("  -", i)

    _mark("completed cell 17")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 17 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 17: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 10.5 — Base-vs-adapter comparison (built into this run)

**Non-negotiable per `docs/LESSONS_LEARNED.md` #1 and `MIGRATION_PLAN.md` P9.**
This is the exact discipline that caught the original Darija
citation-fabrication defect on the Atlas-Chat side: the adapter fabricated
law numbers where the untouched base model correctly said "not in the
text." A trained adapter that fabricates MORE than its own un-adapted base
model on the same grounded prompts is a regression, not an improvement, no
matter how good its loss curve looks.

`model.disable_adapter()` is a context manager on the underlying PEFT model
Unsloth returns — it temporarily turns LoRA off without unloading or
reloading anything, so this costs a few generations, not a second ~15-20 min
model download. Both passes use the same eval rows Section 10 already
picked (grounded, citation-bearing contexts only — a hallucination on an
ungrounded prompt isn't this specific failure mode).

**What this section asserts, and what it only reports:**
- **Hard assert:** the adapter must not cite a reference number absent from
  the source context on a row where the base model did not do so either.
  This is exactly the worst-failure-mode pattern (`green_light_model.md`
  RF10-equivalent for French) — if it fires, do not ship this adapter; the
  training data has a grounding-corrupting slice that needs isolating.
- **Reported, not asserted:** general fluency/quality differences between
  base and adapter — the adapter is *expected* to differ in style, Socratic
  behaviour, citation formatting, etc. Only fabricated-reference regression
  is a hard stop here.


In [ ]:
_mark("entering cell 18")
try:
    # RESUME fix: the original run crashed here with ModuleNotFoundError because
    # app/ was missing from that run's dataset version (see MIGRATION_PLAN.md) --
    # this cell had no HAVE_APP guard even though the smoke-test cell above does.
    # Guarding it the same way closes that gap for any future run, not just this
    # one.
    if not HAVE_APP:
        raise SystemExit(
            "app/ not uploaded -- cannot run the base-vs-adapter citation-"
            "fabrication gate. This is the safety check LESSONS_LEARNED.md #1 "
            "exists for; do not skip it silently. Fix the dataset upload and "
            "re-run rather than shipping an adapter this was never checked "
            "against."
        )

    from app.services.citations import extract_citations
    from app.services.generate_training_data import context_from_system_prompt, row_cites

    # Reuse Section 10's context_from_system_prompt / row_cites-style logic to
    # find real eval rows whose context contains an extractable legal citation
    # -- the same "citable" filter this session's audit used to measure quiz
    # citation recall on data/fr_v3_merged.
    citable_rows = []
    for r in eval_rows:
        if r["component"] not in ("grounded_refusal", "quiz_generation", "socratic"):
            continue
        ctx = context_from_system_prompt(r["messages"][0]["content"])
        anchors = extract_citations(ctx)
        if anchors:
            citable_rows.append((r, ctx, anchors))

    print(f"citable eval rows available for base-vs-adapter comparison: {len(citable_rows)}")
    N_PROBES = min(6, len(citable_rows))
    probes = citable_rows[:N_PROBES]
    assert probes, (
        "No citable eval rows found -- cannot run the base-vs-adapter grounding "
        "check. Do not skip this silently; investigate why eval.jsonl has no "
        "citable grounded_refusal/quiz_generation/socratic rows before shipping."
    )


    def fabricated_refs(answer_text, anchors):
        """Reference numbers the model's own answer cites that do not appear
        among the context's real anchors -- the worst-failure-mode signal."""
        cited = extract_citations(answer_text)
        return set(cited) - set(anchors)


    results = []
    for r, ctx, anchors in probes:
        with model.disable_adapter():
            base_answer = ask(r)
        adapter_answer = ask(r)
        base_fab = fabricated_refs(base_answer, anchors)
        adapter_fab = fabricated_refs(adapter_answer, anchors)
        results.append({
            "component": r["component"],
            "base_answer": base_answer,
            "adapter_answer": adapter_answer,
            "base_fabricated": sorted(str(k) for k in base_fab),
            "adapter_fabricated": sorted(str(k) for k in adapter_fab),
            "regression": bool(adapter_fab) and not base_fab,
        })
        print("=" * 70)
        print(f"[{r['component']}] anchors in context: {sorted(str(k) for k in anchors)}")
        print(f"BASE   fabricated={sorted(str(k) for k in base_fab)}")
        print(f"  {base_answer[:300]}")
        print(f"ADAPTER fabricated={sorted(str(k) for k in adapter_fab)}")
        print(f"  {adapter_answer[:300]}")

    _mark("completed cell 18")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 18 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 18: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 19")
try:
    regressions = [r for r in results if r["regression"]]
    print(f"\nbase-vs-adapter grounding regressions: {len(regressions)} / {len(results)} probes")
    for r in regressions:
        print(f"  - [{r['component']}] adapter fabricated {r['adapter_fabricated']} "
              f"where base fabricated nothing")

    BASE_VS_ADAPTER_REGRESSION = bool(regressions)
    assert not BASE_VS_ADAPTER_REGRESSION, (
        f"{len(regressions)} probe(s) show the adapter fabricating a citation the "
        "un-adapted base model did not, on the same grounded prompt. This is the "
        "exact citation-fabrication regression docs/LESSONS_LEARNED.md #1 warns "
        "about. DO NOT ship this adapter -- isolate which training rows taught "
        "this pattern (see this session's F1/quiz-fabrication audit methodology) "
        "and regenerate that slice before retraining."
    )
    print("\nbase-vs-adapter check PASSED: adapter introduces no new citation "
          "fabrication relative to stock Gemma-2-9B on these grounded probes.")

    _mark("completed cell 19")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 19 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 19: " + type(_e).__name__ + ": " + str(_e))
    raise


## Section 11 — Save, patch, package, export

Priority order matters. `LOCKEDIN_PLAN.md` §2.3 puts production on **vLLM
multi-LoRA**, so the **adapter is the primary artifact** and the GGUF is a
convenience for local Ollama dev. The adapter is therefore saved, patched and
zipped *first*; the GGUF is attempted afterwards behind a disk-space guard.
`save_pretrained_gguf` merges to 16-bit (~18.5 GB) before quantising (~5.8
GB), which does not fit inside `/kaggle/working`'s 20 GB cap, and it compiles
llama.cpp — the least reliable step in this notebook. It must never be the
reason a trained adapter is lost.

In [ ]:
_mark("entering cell 20")
try:
    LORA_DIR = "/kaggle/working/lora_model"
    model.save_pretrained(LORA_DIR)
    tokenizer.save_pretrained(LORA_DIR)   # carries our system-aware chat_template
    print("saved adapter + patched tokenizer ->", LORA_DIR)

    # Gemma-2-9B's generation_config lists only <eos> (1), same as Atlas-Chat
    # (which inherits it). Our rows terminate with <end_of_turn> (107), so
    # unpatched the served model never stops — it just starts role-playing
    # the user. repetition_penalty is also a STRING upstream on some configs,
    # which some loaders reject.
    gc_path = Path(LORA_DIR) / "generation_config.json"
    gen_cfg = json.loads(gc_path.read_text()) if gc_path.exists() else {}
    gen_cfg.update({
        "bos_token_id": BOS_ID,
        "eos_token_id": STOP_IDS,
        "pad_token_id": tokenizer.pad_token_id,
    })
    gen_cfg.pop("repetition_penalty", None)
    gen_cfg.pop("cache_implementation", None)
    gc_path.write_text(json.dumps(gen_cfg, indent=2))
    print("patched generation_config:", gen_cfg)

    # Verify the saved tokenizer really carries the template, not the stock one
    # that raises on system messages.
    from transformers import AutoTokenizer
    _rt = AutoTokenizer.from_pretrained(LORA_DIR)
    _probe = train_rows[0]["messages"]
    assert _rt.apply_chat_template(_probe, tokenize=False) == \
        tokenizer.bos_token + render(_probe), \
        "Saved tokenizer does not reproduce the training format — vLLM would " \
        "serve a different prompt shape than we trained on."
    print("round-trip OK: saved tokenizer reproduces the training format")

    for p in sorted(Path(LORA_DIR).iterdir()):
        print(f"  {p.name:34s} {p.stat().st_size / 1e6:8.2f} MB")
    _mark("completed cell 20")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 20 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 20: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 21")
try:
    # Ollama Modelfile. The TEMPLATE reproduces the training format exactly:
    # system merged into the first user turn, separated by a blank line. llm.py
    # calls /api/generate with `system` + `prompt`, which is what .System and
    # .Prompt bind to, so this is the path production actually takes today.
    # Built by joining lines rather than with an f-string so the Go-template
    # braces need no escaping.
    GGUF_NAME = "iblog-tutor-fr-q4_k_m.gguf"
    Q3 = '"' * 3

    TEMPLATE_BODY = "\n".join([
        "{{ if .System }}<start_of_turn>user",
        "{{ .System }}",
        "",
        "{{ .Prompt }}<end_of_turn>",
        "{{ else }}<start_of_turn>user",
        "{{ .Prompt }}<end_of_turn>",
        "{{ end }}<start_of_turn>model",
        "{{ .Response }}<end_of_turn>",
    ])

    MODELFILE = "\n".join([
        f"FROM ./{GGUF_NAME}",
        "",
        f"TEMPLATE {Q3}{TEMPLATE_BODY}",
        Q3,
        "",
        'PARAMETER stop "<end_of_turn>"',
        'PARAMETER stop "<start_of_turn>"',
        "PARAMETER temperature 0.2",
        "PARAMETER num_ctx 4096",
        "",
    ])
    Path("/kaggle/working/Modelfile").write_text(MODELFILE, encoding="utf-8")
    print(MODELFILE)
    _mark("completed cell 21")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 21 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 21: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 22")
try:
    report = {
        "language": "fr",
        "dataset": "fr_v3_merged",
        "base_model": BASE_MODEL,
        "gpu": GPU_NAME,
        "precision": "bf16" if SUPPORTS_BF16 else "fp16",
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora": {
            "r": 16, "alpha": 16, "dropout": 0,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                               "gate_proj", "up_proj", "down_proj"],
        },
        "trainable_params": trainable,
        "epochs": EPOCHS,
        "effective_batch": BATCH * ACCUM,
        "learning_rate": LR,
        "scheduler": "cosine",
        "rows": {"train": len(train_ds), "eval": len(eval_ds),
                 "dropped_overlong_frac": round(drop_rate, 5)},
        "trainable_tokens_per_epoch": total_target_tokens,
        "train_minutes": train_minutes,
        "steps_completed": steps_completed,
        "steps_planned": steps_planned,
        "eval_loss_first_recorded": (
            round(first_recorded_eval_loss, 5) if first_recorded_eval_loss is not None else None
        ),
        "eval_loss_last_recorded": (
            round(final_recorded_eval_loss, 5) if final_recorded_eval_loss is not None else None
        ),
        "train_loss_final_logged": (
            round(final_train_loss, 5) if final_train_loss is not None else None
        ),
        "eval_curve": [{"step": h.get("step"), "eval_loss": round(h["eval_loss"], 5)}
                       for h in hist],
        "best_eval_step": (best.get("step") if best is not None else None),
        "per_component_eval_loss": {
            c: {"train_rows": tr, "eval_rows": ev, "loss_per_token": round(l, 5)}
            for c, tr, ev, l in rows_out
        },
        "overall_eval_loss_per_token": round(overall, 5),
        "system_merge": {"strategy": "system merged into first user turn",
                         "separator": SYSTEM_JOIN},
        "stop_token_ids": STOP_IDS,
        "prompt_parity_checked": PARITY_CHECKED,
        "smoke_test_issues": issues,
        "base_vs_adapter": {
            "probes_run": len(results),
            "regressions": len(regressions),
            "regression_detail": [
                {"component": r["component"], "adapter_fabricated": r["adapter_fabricated"]}
                for r in regressions
            ],
            "passed": not BASE_VS_ADAPTER_REGRESSION,
        },
        "resumed_from_checkpoint": "checkpoint-174",
        "resume_note": (
            "The original run (kernel darija-tutor-fr-finetune-v1) trained this "
            "adapter to completion (174/174 steps, 2/2 epochs) but crashed "
            "afterward with ModuleNotFoundError: no module named 'app' -- the "
            "Kaggle dataset version it pulled from was missing app/ at run time "
            "(uploaded correctly before and after, not during -- see "
            "MIGRATION_PLAN.md). This run resumed from that adapter without "
            "retraining and re-ran every post-training check that never got a "
            "chance to execute: per-component eval, sample generation, the "
            "smoke test, and the base-vs-adapter fabrication gate. "
            "eval_loss_first_recorded/eval_loss_last_recorded are the earliest "
            "and latest points HF Trainer actually logged before the crash, not "
            "a formal pre-training baseline (that ad-hoc trainer.evaluate() call "
            "was never persisted to trainer_state.json)."
        ),
    }
    Path("/kaggle/working/TRAINING_REPORT.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))

    _mark("completed cell 22")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 22 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 22: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 23")
try:
    # Zip the adapter FIRST — before the GGUF step, which is the flaky one.
    export_dir = Path("/kaggle/working/finetuned_model_export")
    if export_dir.exists():
        shutil.rmtree(export_dir)
    export_dir.mkdir()
    shutil.copytree(LORA_DIR, export_dir / "lora_model")
    shutil.copy("/kaggle/working/Modelfile", export_dir / "Modelfile")
    shutil.copy("/kaggle/working/TRAINING_REPORT.json",
                export_dir / "TRAINING_REPORT.json")
    (export_dir / "chat_template.jinja").write_text(CHAT_TEMPLATE, encoding="utf-8")

    shutil.make_archive("/kaggle/working/finetuned_model_export", "zip",
                        str(export_dir))
    z = Path("/kaggle/working/finetuned_model_export.zip")
    print(f"PACKAGED {z} — {z.stat().st_size / 1e6:.1f} MB")
    print("This zip is the deliverable. Everything below is optional.")
    _mark("completed cell 23")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 23 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 23: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 24")
try:
    # GGUF is best-effort by design: merging to 16-bit needs ~18.5GB and llama.cpp
    # has to be compiled. The adapter above is already safe on disk.
    GGUF_BUILD = "/root/gguf_build"     # container disk, NOT the 20GB working quota
    free_root = shutil.disk_usage("/")[2] / 1e9
    free_work = shutil.disk_usage("/kaggle/working")[2] / 1e9
    print(f"free: container {free_root:.1f} GB, working {free_work:.1f} GB "
          f"(need ~26 / ~7)")

    if free_root < 26 or free_work < 7:
        print("SKIPPING GGUF export — not enough disk. Rebuild locally from the "
              "adapter zip; see FINETUNE_AND_DEPLOY.md.")
    else:
        try:
            t_g = time.time()
            model.save_pretrained_gguf(GGUF_BUILD, tokenizer,
                                       quantization_method="q4_k_m")
            produced = sorted(Path(GGUF_BUILD).rglob("*.gguf"),
                              key=lambda p: p.stat().st_size, reverse=True)
            print(f"gguf build took {(time.time() - t_g) / 60:.1f} min; found "
                  f"{[p.name for p in produced]}")
            if produced:
                dst = Path("/kaggle/working") / GGUF_NAME
                shutil.move(str(produced[0]), dst)
                print(f"GGUF -> {dst} ({dst.stat().st_size / 1e9:.2f} GB)")
                print("Left unzipped on purpose: a q4 GGUF does not compress, and "
                      "zipping it would need another ~6GB of the working quota.")
            shutil.rmtree(GGUF_BUILD, ignore_errors=True)
        except Exception as e:
            print(f"GGUF export FAILED ({type(e).__name__}: {e})")
            print("Newer Unsloth builds renamed the kwarg to quantization_type — "
                  "if that is the error, retry with that name.")
            print("The adapter zip is unaffected; rebuild the GGUF locally.")
            shutil.rmtree(GGUF_BUILD, ignore_errors=True)
    _mark("completed cell 24")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 24 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 24: " + type(_e).__name__ + ": " + str(_e))
    raise


In [ ]:
_mark("entering cell 25")
try:
    print("=" * 66)
    print("ARTIFACTS IN /kaggle/working")
    print("=" * 66)
    for p in sorted(Path("/kaggle/working").iterdir()):
        if p.is_file():
            print(f"  {p.name:44s} {p.stat().st_size / 1e6:9.1f} MB")
        else:
            print(f"  {p.name + '/':44s} (dir)")

    print()
    print("Primary  : finetuned_model_export.zip")
    print("           (LoRA adapter + patched tokenizer/chat_template +")
    print("            Modelfile + TRAINING_REPORT.json)")
    print(f"Secondary: {GGUF_NAME}, if the GGUF step ran")
    print(f"Checkpoints: /kaggle/working/checkpoints (per-epoch, for rollback)")
    print()
    print("SUMMARY")
    print(f"  eval_loss {report['eval_loss_before']} -> {report['eval_loss_after']}")
    print(f"  steps {report['steps_completed']}/{report['steps_planned']}"
          f"  in {report['train_minutes']} min")
    print(f"  smoke-test issues: {len(issues)}")
    print(f"  base-vs-adapter: {report['base_vs_adapter']['regressions']}/"
          f"{report['base_vs_adapter']['probes_run']} regressions "
          f"({'PASS' if report['base_vs_adapter']['passed'] else 'FAIL'})")
    print()
    print("Next: read TRAINING_REPORT.json, then FINETUNE_AND_DEPLOY.md ->")
    print("  'Go/no-go before deploying' and 'Train/serve parity'.")
    _mark("completed cell 25")
except BaseException as _e:
    import traceback
    with open(_CRASH_LOG, "a", encoding="utf-8") as _f:
        _f.write("\n=== CRASH in cell 25 (" + type(_e).__name__ + ") " + _t.strftime("%H:%M:%S") + " ===\n")
        _f.write(traceback.format_exc())
    _mark("CRASHED in cell 25: " + type(_e).__name__ + ": " + str(_e))
    raise
